# PyTorch
### 1)tensors

In [1]:
import torch
torch.cuda.is_available()

True

In [99]:
from torch import tensor

## Shape & Structure (MOST IMPORTANT for CNNs)
#### tensor.Size([batch_size, channels, height, width])
#### tensor.ndim = no of dimensions | 0D = scalar | 1D = vector | 2D = matrix | 3D = tensor
#### tensor.size()
#### tensor.numel() = elements present in the tensor | Used before flattening

## Data Type & Precision
#### tensor.dtype = Data type of tensor | CNN rules of thumb: Images → float32 & Labels → int64 (for CrossEntropyLoss)
#### tensor.is_floating_point() = Convolutions & gradients require floating-point tensors.

## Device & Memory Placement
#### tensor.device = CPU or GPU | Critical when moving to GPU
#### tensor.is_cuda = Common error: RuntimeError: Tensor on CPU and model on CUDA

## Gradient & Autograd (Backprop Core) | This is where learning actually happens.
#### tensor.requires_grad = True → gradients will be computed & Model weights → True & Input images → usually False
#### tensor.grad = Stores gradient after backprop | Debugging vanishing/exploding gradients & Custom training loops
#### tensor.grad_fn = Shows how tensor was created (computation graph) | Helpful for understanding backprop flow.

## Layout, Contiguity & Reshaping | Very important when reshaping CNN outputs
#### tensor.is_contiguous() = After transpose or permute, tensors are often non-contiguous.
#### tensor.stride() = How tensor is laid out in memory | Used for deep debugging and performance optimization.

## Statistical & Value Inspection | Used during debugging and normalization. | Are images normalized? Is output exploding?
#### tensor.min()
#### tensor.max()
#### tensor.mean() not applicable on scaler tensors
#### tensor.std() not applicable on scaler tensors
#### tensor.sum() = Used in loss calculations and sanity checks.

## Conversion & Interoperability
#### tensor.item() = Extract Python scalar (single element tensor)
#### tensor.numpy() = Convert to NumPy (CPU only) | Breaks autograd graph.

## Indexing & Slicing (Data Inspection) | Used for visualizing images or debugging batches.
#### tensor[0]          # first image
#### tensor[:, 0]       # first channel
#### tensor[0, :, :, :] # explicit

## Type & Identity Checks
#### tensor.type() = 
#### tensor.is_leaf = Whether the tensor is a leaf node in the computation graph | Important for understanding why .grad may be None.


In [13]:
# 0D
scaler = tensor(1)
print(f'shape={scaler.shape} |  dimensions={scaler.ndim} | size={scaler.size()} | no of elements={scaler.numel()}')

shape=torch.Size([]) |  dimensions=0 | size=torch.Size([]) | no of elements=1


In [18]:
print(f'elements_datatype={scaler.dtype} |  is_floating_point()={scaler.is_floating_point()} | device={scaler.device} | is_cuda={scaler.is_cuda}')

elements_datatype=torch.int64 |  is_floating_point()=False | device=cpu | is_cuda=False


In [21]:
print(f'require_grad={scaler.requires_grad} |  grad={scaler.grad} | size={scaler.size()} | grad_fn={scaler.grad_fn}')

require_grad=False |  grad=None | size=torch.Size([]) | grad_fn=None


In [28]:
print(f'is_contiguous={scaler.is_contiguous()} |  min={scaler.min()} | max={scaler.max()} | sum={scaler.sum()}')

is_contiguous=True |  min=1 | max=1 | sum=1


In [72]:
# index of minimum and maximum
tensor1 = torch.tensor([1,2,2,3,5,6,9,8,74])
print(f'index of min element is: {tensor1.argmin()}, index of max element is: {tensor1.argmax()}')

index of min element is: 0, index of max element is: 8


In [37]:
print(f'items={scaler.item()} |  numpy={scaler.numpy()}')

items=1 |  numpy=1


In [40]:
print(f'is_leaf={scaler.is_leaf} |  type={scaler.type()}')

is_leaf=True |  type=torch.LongTensor


| Attribute       | Why it matters                        |
| --------------- | ------------------------------------- |
| `shape`         | Channel ordering, layer compatibility |
| `dtype`         | Loss functions & gradients            |
| `device`        | CPU vs GPU errors                     |
| `requires_grad` | Learning vs frozen layers             |
| `is_contiguous` | `.view()` vs `.reshape()`             |
| `grad`          | Debug training                        |
| `ndim`          | Conv vs Linear layers                 |


# Practice with example | CNN kernels operate per channel, not per pixel.

# Why this matters : 1)When debugging: Wrong accuracy → inspect individual images & Strange output → check channel order

In [66]:
# Image batch: 2 images, 3 channels, 32x32
x = torch.randn(2, 3, 32, 32)

In [67]:
x.shape

torch.Size([2, 3, 32, 32])

# CNNs are extremely sensitive to tensor shape and dimension order.
# Different libraries and operations expect data in different formats:
# Format name	Order
## NCHW (PyTorch default)	(batch, channels, height, width)
## NHWC (TensorFlow common)	(batch, height, width, channels)
### If your tensor is in the wrong order:
#### 1)Convolution will fail
#### 2)Or worse, it will run but learn wrong features

In [68]:
y=x.permute(0,2,3,1)
# What changed?
# Dimension 0 stayed the same
# Dimension 1 moved to position 3
# Dimension 2 moved to position 3
# Dimension 3 moved to position 4
y.shape

torch.Size([2, 32, 32, 3])

In [56]:
# x[0]       # first image → shape (3, 32, 32)
# x[0, 0]     # red channel of first image → (32, 32)
len(x[:, 0])     # red channel for all images → (2, 32, 32)


2

### You’ll routinely use permute when:
#### 1.Writing custom datasets
#### 2.Mixing NumPy / OpenCV / PIL with PyTorch
#### 3.Working with attention maps
#### 4.Implementing Vision Transformers
#### 5.Visualizing feature maps
#### 6.Switching between CNN ↔ RNN / Transformer

# `view` vs `reshape` vs `permute` (CRITICAL) | This is where most CNN `bugs happen`.

| Function  | Purpose            | Changes data order?    | Needs contiguous memory? |
| --------- | ------------------ | ---------------------- | ------------------------ |
| `permute` | Reorder dimensions | ❌ No (only axes order) | ❌ No                     |
| `view`    | Reshape tensor     | ❌ No                   | ✅ Yes                    |
| `reshape` | Reshape tensor     | ❌ No*                  | ❌ No*                    |


# permute – Change axis order | After permute, tensor becomes non-contiguous:
#### What it does
#### Rearranges dimensions
#### Does NOT move data in memory
#### Returns a view with new stride order
#### Think of it as: “I want to look at the same data, but interpret axes differently.”

# view – Reshape WITHOUT copying data | CNN Example: Flatten before FC layer & Used all the time before nn.Linear
#### What it does
#### Changes tensor shape
#### No data movement
#### Requires contiguous memory
#### Think of it as: "Cut the same memory block into a new shape.”

# reshape – Smart version of view | 
#### What it does
#### Reshapes tensor
#### If contiguous → behaves like view
#### If non-contiguous → creates a copy
#### Think of it as: “Reshape it — I don’t care how.”

| Case           | Behavior    |
| -------------- | ----------- |
| Contiguous     | No copy     |
| Non-contiguous | Copies data |


# When to use what (rule of thumb)
#### ✅ Use permute when:
#### 1: Changing dimension order (HWC ↔ CHW)
#### 2: CNN ↔ Transformer
#### 3: Visualization

#### ✅ Use view when:
#### 1: Tensor is contiguous
#### 2: Performance is critical
#### 3: Flattening CNN outputs

####  ✅ Use reshape when:
#### 1: You’re unsure about contiguity
#### 2: Writing safe / clean code
#### 3:Rapid prototyping

# 🧠 One-line intuition summary
#### permute → “Change axis order”
#### view → “Reshape without copying”
#### reshape → “Reshape no matter what”

# lets create tensors
# 1. torch.tensor

In [82]:
vector = tensor([1,1]) # can also be called array 
vector.dim(), vector.shape

(1, torch.Size([2]))

In [90]:
matrix = tensor([[1,2,3],[1,1,1]])
matrix, matrix.dim(), matrix.shape

(tensor([[1, 2, 3],
         [1, 1, 1]]),
 2,
 torch.Size([2, 3]))

In [108]:
tensor1 = tensor([
    [
        [1,2],
        [3,4]
    ],
    [
        [1,2],
        [3,4]
    ],
    [
        [1,2],
        [3,4]
    ]
])
tensor1, tensor1.dim(), tensor1.shape

(tensor([[[1, 2],
          [3, 4]],
 
         [[1, 2],
          [3, 4]],
 
         [[1, 2],
          [3, 4]]]),
 3,
 torch.Size([3, 2, 2]))

| Feature     | `torch.tensor`    | `torch.rand`   |
| ----------- | ----------------- | -------------- |
| Input data  | Required          | Not required   |
| Random      | ❌ No              | ✅ Yes          |
| Copies data | ✅ Yes             | N/A            |
| Used for    | Constants, labels | Initialization |


#  2. via torch.rand

In [120]:
# can not directly create 0D we can squeeze a 1D tensor like
scalerx = torch.rand(1)
scaler = scalerx.squeeze()
scaler

tensor(0.2130)

### 1D tensor | simple array, i.e., A single list, having 5 elements in it
| Dimension | Value | Meaning                    |
| --------- | ----- | -------------------------- |
| `H`       | 5     | length of array            |


In [121]:
# 1D tensor | simple array of 5 elements
vector = torch.rand(5)
vector

tensor([0.6787, 0.6734, 0.5985, 0.5649, 0.8408])

### 2D tensor | simple single images, i.e., A single image, having 1 channel Black and white or grayscale, where the frame is of size 3×3 pixels
| Dimension | Value | Meaning                    |
| --------- | ----- | -------------------------- |
| `H`       | 3     | Frame height               |
| `W`       | 3     | Frame width                |


In [134]:
matrix = torch.rand(3,3)   # height, width
matrix

tensor([[0.0804, 0.0706, 0.4979],
        [0.6076, 0.6587, 0.4087],
        [0.5144, 0.3230, 0.1217]])

### 3D tensor | simple single images, i.e., A single image, having 3 channels RGB, where the frame is of size 20×20 pixels
| Dimension | Value | Meaning                    |
| --------- | ----- | -------------------------- |
| `C`       | 3     | **3 color channels (RGB)** |
| `H`       | 20   | Frame height               |
| `W`       | 20   | Frame width                |


In [132]:
image = torch.rand(3,20,20)   # channels, height, width
image.shape

torch.Size([3, 20, 20])

### 4D tensor | collection of images / Video, i.e., A batch of 10 images, each image having 3 channels RGB, where each frame is a 20×20 image
| Dimension | Value | Meaning                    |
| --------- | ----- | -------------------------- |
| `N`       | 2     | **2 separate images**      |
| `C`       | 3     | **3 color channels (RGB)** |
| `H`       | 20   | Frame height               |
| `W`       | 20   | Frame width                |


In [131]:
batch_of_images = torch.rand(10,3,20,20)  # batch size, channels, height, width
batch_of_images.shape

torch.Size([10, 3, 20, 20])

### 5D tensor | batch of videos/ Volume/ VideoS, i.e., A batch of 2 videos, each video having 16 RGB frames, where each frame is a 224×224 image
| Dimension | Value | Meaning                    |
| --------- | ----- | -------------------------- |
| `N`       | 2     | **2 separate videos**      |
| `C`       | 3     | **3 color channels (RGB)** |
| `D`       | 16    | **16 frames per video**    |
| `H`       | 224   | Frame height               |
| `W`       | 224   | Frame width                |


In [130]:
batch_of_video = torch.rand(2,3,16,224,224)
batch_of_video.shape

torch.Size([2, 3, 16, 224, 224])

|0D | scalar        → loss
|1D |→ vector        → bias, labels
|2D |→ matrix        → FC weights, grayscale image
|3D |→ image         → (C, H, W)
|4D |→ batch         → (N, C, H, W)
|5D → video/volumes → (N, C, D, H, W)


# summary
| Dimension   | LITERAL MEANING | EXAMPLE                    |short hand |
| ----------- | --------------- | -------------------------- |-----------|
| `'0D'`      | scalar          | loss                       | lower(`a`)  |
| `'1D'`      | vector          | bias, labels               |lower(`y`)   |
| `'2D'`      | matrix          | FC weights, grayscale image|upper(`Q`)   |
| `'3D'`      | image           | (C, H, W)                  |upper(`X`)   |
| `'4D'`      | batch           | (N, C, H, W)               |upper(`X`)   |
| `'5D'`      | video/volumes   | (N, C, D, H, W)            |upper(`X`)   |

# examples
### 1) tensor generation

In [27]:
import torch
# torch.cuda.is_available()

In [28]:
# zero tensor
zero_tensor = torch.zeros(1,3,5,5, dtype=torch.float64, device='cpu')
zero_tensor

tensor([[[[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]]]], dtype=torch.float64)

In [35]:
# ones tenosr
one_tensor = torch.ones(1,2,3,3)
one_tensor

tensor([[[[1., 1., 1.],
          [1., 1., 1.],
          [1., 1., 1.]],

         [[1., 1., 1.],
          [1., 1., 1.],
          [1., 1., 1.]]]])

In [3]:
# torch.range deprocated use newer one

In [44]:
# torch.arange
ranged_tensor0 = torch.arange(10) # starting from 0 to n 
ranged_tensor1 = torch.arange(1,10) # starting from P to Q 
ranged_tensor2 = torch.arange(0,10,2) # with step 2
ranged_tensor0, ranged_tensor1,ranged_tensor2,

(tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 tensor([1, 2, 3, 4, 5, 6, 7, 8, 9]),
 tensor([0, 2, 4, 6, 8]))

In [5]:
# tensor alike

In [49]:
# torch.tensor learn parameters
a = torch.tensor(data=(1,1), dtype=torch.float16, device='cpu', requires_grad=False, pin_memory=False) # layout for faster CPU <-> GPU copy
a

tensor([1., 1.], dtype=torch.float16)

#### In PyTorch, requires_grad is a core switch for automatic differentiation. | It tells PyTorch whether to track operations on a tensor so that gradients can be computed during backpropagation.

# OPERATION ON TENSORS

In [56]:
# addition
tensor1 = torch.tensor([10,10])
tensor2 = torch.tensor(2)
addition = tensor1 + tensor2
addition, torch.add(tensor1,tensor2)
# addition is of 2 types 1:scalar addition like above, 2: matrix row wise addition where shape matters

(tensor([12, 12]), tensor([12, 12]))

In [57]:
# substraction
# operation samew as above , most common use in calculating errors (mse etc)

In [59]:
#  multiplication (element wise) SCALER * VECTOR
tensor1 = torch.tensor([10,10])
tensor2 = torch.tensor(2)
multiplication = tensor1 * tensor2
multiplication, torch.mul(tensor1,tensor2)

(tensor([20, 20]), tensor([20, 20]))

In [62]:
#  multiplication: MATRIX * VECTOR
tensor1 = torch.tensor([[1,2],[3,4]])
tensor2 = torch.tensor([2,3])
multiplication = tensor1 * tensor2
multiplication

tensor([[ 2,  6],
        [ 6, 12]])

In [65]:
#  multiplication: MATRIX * matrix  # 
tensor1 = torch.tensor([[1,2],[3,4]])
tensor2 = torch.tensor([[5,6],[7,8]])
multiplication = tensor1 @ tensor2
multiplication

tensor([[19, 22],
        [43, 50]])

In [67]:
#  multiplication: MATRIX * matrix   
# DO NOT MIX WITH CONVOLUTION: WHICH IS MATRIX MULTIPLICATION OF KERNELS, NOT LIKE SIMPLE MULTIPLICATION, WHICH IS MULTIPLICATION OF 2 COMPATIBLE MATRICES
# HOWEVER, convolution is also a linear operation
tensor1 = torch.tensor([[1,2],[3,4]])
tensor2 = torch.tensor([[5,6],[7,8]])
multiplication = torch.matmul(tensor1, tensor2)
multiplication

tensor([[19, 22],
        [43, 50]])

In [11]:
# division use in normalization
# operations are simple as above

In [71]:
# transpose
tensor1 = torch.tensor([[1,2],[3,4]])
transpose = tensor1.T
transpose

tensor([[1, 3],
        [2, 4]])

In [73]:
# reshapin

In [74]:
# stacking

In [77]:
# squeezing

In [78]:
# permute

# DATA FETCHING FROM TENSOR

In [88]:
# indexing:1
# similat to numpy
tensor1 = torch.arange(1,10).reshape(1,3,3)
tensor1, tensor1[0],tensor1[0][0],tensor1[0][2][2]

(tensor([[[1, 2, 3],
          [4, 5, 6],
          [7, 8, 9]]]),
 tensor([[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]),
 tensor([1, 2, 3]),
 tensor(9))

In [95]:
# indexing:2
# select all from row/dimension (':' colon referring to all from the index that it is used)
# select all from zeroth index, i.e., full matrix, and then zeroth element from dimension one, i.e., [1,2,3,]
print(f'all from zeroth dimnestion= {tensor1[:,0]}')

all from zeroth dimnestion= tensor([[1, 2, 3]])


In [103]:
# indexing 3
# selecting column from array
# select all from zeroth index, i.e., full matrix, and then all axes(columns), and then mention the index no. of the column
# e.g select all elements of the second column (2nd column indexing is 1 here)
print(f'all from zeroth dimnestion= {tensor1[:,:,1]}')

all from zeroth dimnestion= tensor([[2, 5, 8]])


# TENSOR DATA CONVERSION

In [ ]:
# numpy and tensors
# 1 from numpy array to tensor

In [ ]:
# 2 from tensor to numpy

# REPRODUCIBILITY
#### ELIMINATING RANDOMNESS FROM RANDOM
#### WORKING OF CNN:
#### `RANDOM NO INITIALIZATION -> TENSOR OPERATIONS -> UPDATE(MULTIPLE TIMES) NUMBERS TO FIT THE REPRESENTATION OF DATA`
#### WE CAN CONTROL THIS RANDOMNESS OF THE PRIMARILY INITIALIZED WEIGHTS


In [113]:
#RANDOM_SEED / manual seeding
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
tensor1= torch.rand(3,3)
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
tensor2= torch.rand(3,3)

print(f'tensor1={tensor1}, \n tensor2={tensor2},\n {tensor1==tensor2}')


tensor1=tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009],
        [0.2566, 0.7936, 0.9408]]), 
 tensor2=tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009],
        [0.2566, 0.7936, 0.9408]]),
 tensor([[True, True, True],
        [True, True, True],
        [True, True, True]])


In [121]:
# to write device agnostic code, use the following setup always to update in global config
availability = "GPU & CPU both are available" if torch.cuda.is_available() else "CPU available only."
availability

'CPU available only.'